# آموزش مدل تشخیص آلزایمر از روی MRI — GSAH

این نوت‌بوک برای اجرا در **Google Colab** با پردازنده گرافیکی (GPU) طراحی شده است.

**قبل از شروع:** از منوی بالا `Runtime → Change runtime type → T4 GPU` را انتخاب کنید.

## مدل انتخابی: MobileNetV2 (Transfer Learning)

برای این مسئله (طبقه‌بندی تصاویر MRI مغز به ۴ رده) از **MobileNetV2** با وزن‌های از پیش آموزش‌دیده روی ImageNet استفاده می‌کنیم، به دلایل زیر:

- **سبک و سریع**: حدود ۳.5 میلیون پارامتر — روی GPU رایگان Colab (T4) هر epoch چند ثانیه تا چند دقیقه طول می‌کشد.
- **دقت مناسب**: در مسائل مشابه طبقه‌بندی MRI آلزایمر با روش Transfer Learning معمولاً به دقت ۹۰-۹۸٪ می‌رسد؛ کاملاً برای این استفاده کافی است.
- **سبک برای دیپلوی**: خروجی نهایی (`.keras`) کوچک است و به‌راحتی روی سرور Django (حتی بدون GPU) قابل اجراست — نکته مهم چون سرور پروژه شما GPU ندارد.
- در مقایسه، مدل‌های سنگین‌تر مثل ResNet50 یا EfficientNetB3 دقت کمی بالاتر می‌دهند ولی حجم و زمان استنتاج (inference) روی CPU سرور را چند برابر می‌کنند — برای این پروژه به‌صرفه نیست.

> اگر بعداً خواستید دقت را کمی بالاتر ببرید، فقط کافیست در سلول ساخت مدل `MobileNetV2` را با `EfficientNetB0` جایگزین کنید؛ باقی خط لوله (pipeline) بدون تغییر کار می‌کند.

## دیتاست

از دیتاست معروف و عمومی Kaggle به نام **"Alzheimer's Dataset (4 class of Images)"** استفاده می‌کنیم:
`tourist55/alzheimers-dataset-4-class-of-images`

شامل ۴ رده: `NonDemented`, `VeryMildDemented`, `MildDemented`, `ModerateDemented` — حدود ۶۴۰۰ تصویر MRI برش‌خورده (slice)، از پیش به Train/Test تقسیم شده.

> دیتاست بالینی ADNI (که در فاز پروژه به آن اشاره شده) دقیق‌تر و معتبرتر است ولی نیاز به ثبت‌نام و تأیید دسترسی دارد و در قالب NIfTI سه‌بعدی است (پردازش پیچیده‌تر). دیتاست Kaggle بالا برای نمونه اولیه (MVP) و این ساختار پروژه بهترین گزینه عملی است. اگر بعداً دسترسی ADNI را گرفتید، فقط سلول دانلود داده و پیش‌پردازش را عوض کنید — بقیه خط لوله همان می‌ماند.

## خروجی نهایی این نوت‌بوک

دو فایل که باید در مسیر `apps/diagnosis/ml/` پروژه جنگو کپی شوند:

- `alzheimer_model.keras` — مدل آموزش‌دیده
- `class_names.json` — ترتیب دقیق کلاس‌ها (برای اینکه Django بداند هر خروجی مدل مربوط به کدام رده است)

کد سمت جنگو (`apps/diagnosis/ml_inference.py`) از قبل این دو فایل را با همین نام‌ها و همین ساختار می‌خواند — کافیست فایل‌ها را کپی کنید و `DEMO_MODE = False` را در `core/settings.py` تنظیم کنید.


## ۱. نصب و بررسی پیش‌نیازها

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)

gpus = tf.config.list_physical_devices('GPU')
print("GPU available:", gpus)
assert len(gpus) > 0, "GPU پیدا نشد! از منوی Runtime > Change runtime type > GPU (T4) را فعال کنید."


In [ ]:
!pip install -q kagglehub


## ۲. دانلود دیتاست از Kaggle

برای دانلود دیتاست دو راه دارید — هر کدام راحت‌تر بود همان را اجرا کنید (سلول بعدی هر دو را پشتیبانی می‌کند):

**روش ۱ — فایل `kaggle.json` (پیشنهادی، بدون نیاز به لاگین مرورگر):**
1. وارد حساب Kaggle خودتان شوید → `Settings` → بخش `API` → دکمه `Create New Token`.
2. یک فایل به نام `kaggle.json` دانلود می‌شود (شامل `username` و `key`).
3. سلول زیر را اجرا کنید و همان فایل را آپلود کنید.

**روش ۲ — ورود مستقیم (Login تعاملی):**
اگر فایل `kaggle.json` ندارید، سلول زیر را رد کنید (Skip) و سلول دانلود، خودش یک لینک ورود به حساب Kaggle نشان می‌دهد.

In [ ]:
import os

# روش ۱: اگر فایل kaggle.json دارید، این سلول را اجرا کنید تا آپلود شود.
# اگر ندارید، این سلول را رد کنید و مستقیم به سلول بعدی (دانلود دیتاست) بروید —
# kagglehub خودش لینک ورود تعاملی (Login) نشان می‌دهد.

from google.colab import files

os.makedirs("/root/.kaggle", exist_ok=True)
uploaded = files.upload()  # فایل kaggle.json را اینجا انتخاب کنید

if "kaggle.json" in uploaded:
    with open("/root/.kaggle/kaggle.json", "wb") as f:
        f.write(uploaded["kaggle.json"])
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    # kagglehub این مسیر را به‌صورت خودکار به‌جای لاگین تعاملی استفاده می‌کند
    print("kaggle.json ذخیره شد — نیازی به Login تعاملی نیست.")
else:
    print("فایلی آپلود نشد — سلول بعدی از روش Login تعاملی استفاده خواهد کرد.")


In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("tourist55/alzheimers-dataset-4-class-of-images")
print("Dataset downloaded to:", dataset_path)

import os
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    if level <= 2:
        print('  ' * level + os.path.basename(root) + '/')


In [ ]:
import glob

# مسیر دقیق پوشه‌های Train/Test داخل دیتاست دانلود شده را پیدا می‌کنیم
train_dir = glob.glob(os.path.join(dataset_path, '**', 'train'), recursive=True)
test_dir  = glob.glob(os.path.join(dataset_path, '**', 'test'), recursive=True)
if not train_dir:
    train_dir = glob.glob(os.path.join(dataset_path, '**', 'Train'), recursive=True)
if not test_dir:
    test_dir = glob.glob(os.path.join(dataset_path, '**', 'Test'), recursive=True)

TRAIN_DIR = train_dir[0]
TEST_DIR  = test_dir[0]
print("TRAIN_DIR:", TRAIN_DIR)
print("TEST_DIR: ", TEST_DIR)


## ۳. آماده‌سازی داده

- اندازه تصویر را روی `224x224` (ورودی استاندارد MobileNetV2) تنظیم می‌کنیم.
- ۱۰٪ از داده Train برای اعتبارسنجی (validation) کنار گذاشته می‌شود.
- پوشه Test به‌عنوان تست نهایی/مستقل استفاده می‌شود.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.1,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.1,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    shuffle=False,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
)

CLASS_NAMES = train_ds.class_names
print("Classes (alphabetical order used by Keras):", CLASS_NAMES)


In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# استخراج لیبل تمام نمونه‌های Train برای محاسبه class weights (چون دیتاست بین کلاس‌ها نامتوازن است)
train_labels = np.concatenate([y.numpy() for _, y in train_ds])
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels,
)
class_weight = {i: w for i, w in enumerate(class_weights_arr)}
print("Class weights:", dict(zip(CLASS_NAMES, class_weights_arr)))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.cache().prefetch(buffer_size=AUTOTUNE)


## ۴. افزایش داده (Augmentation) و پیش‌پردازش

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="augmentation")

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input


## ۵. ساخت مدل (MobileNetV2 + Transfer Learning)

نکته مهم برای سازگاری با Django: نام لایه پایه را `mobilenetv2_base` می‌گذاریم و آخرین لایه کانولوشنی آن `Conv_1` است —
کد Grad-CAM سمت جنگو (`apps/diagnosis/ml_inference.py`) دقیقاً همین نام‌ها را جستجو می‌کند. اگر این نام‌ها را عوض کنید باید آن فایل را هم به‌روز کنید.

In [ ]:
NUM_CLASSES = len(CLASS_NAMES)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base_model._name = "mobilenetv2_base"
base_model.trainable = False  # فاز اول: پایه فریز شده

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs, name="alzheimer_mobilenetv2")
model.summary()


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

EPOCHS_PHASE1 = 12
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    class_weight=class_weight,
    callbacks=callbacks,
)


## ۶. فاین‌تیون (Fine-tuning)

حالا لایه‌های بالایی MobileNetV2 را باز (unfreeze) می‌کنیم و با نرخ یادگیری بسیار پایین‌تر ادامه می‌دهیم تا مدل روی ویژگی‌های خاص تصاویر MRI تنظیم دقیق‌تری پیدا کند.

In [ ]:
base_model.trainable = True

# فقط ۳۰ لایه آخر باز شوند؛ بقیه فریز بمانند تا از overfitting جلوگیری شود
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

EPOCHS_PHASE2 = 10
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    class_weight=class_weight,
    callbacks=callbacks,
)


## ۷. ارزیابی مدل

In [ ]:
import matplotlib.pyplot as plt

def plot_history(h1, h2, key):
    vals = h1.history[key] + h2.history[key]
    val_vals = h1.history[f"val_{key}"] + h2.history[f"val_{key}"]
    plt.plot(vals, label=f"train_{key}")
    plt.plot(val_vals, label=f"val_{key}")
    plt.axvline(len(h1.history[key]) - 1, color="gray", linestyle="--", label="fine-tune start")
    plt.legend()
    plt.title(key)
    plt.show()

plot_history(history1, history2, "accuracy")
plot_history(history1, history2, "loss")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns

test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc*100:.2f}%")

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred_probs = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
print(f"F1 (macro):    {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"F1 (weighted): {f1_score(y_true, y_pred, average='weighted'):.4f}")

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True"); axes[0].set_title("Confusion Matrix (counts)")

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True"); axes[1].set_title("Confusion Matrix (row-normalized)")
plt.tight_layout()
plt.show()


> **⚠️ محدودیت مهم درباره اعتبار این عدد دقت:**
>
> این دیتاست (و عملاً همه نسخه‌های عمومی این دیتاست روی Kaggle، از جمله نسخه‌های "augmented" با ادعای دقت ۹۹٪) شناسه بیمار (patient ID) ندارد.
> یعنی ممکن است چند برش (slice) از اسکن MRI **یک بیمار** هم در Train و هم در Test پخش شده باشد، پس دقت گزارش‌شده در واقع
> **دقت روی تصویر (image-level)** است، نه دقت روی **بیمار جدید که مدل قبلاً ندیده (patient-level)**.
>
> نسخه‌های "augmented" دیگر این دیتاست (مثل نسخه‌ای که ادعای ۹۹٪ دقت دارد) این مشکل را بدتر هم می‌کنند، چون augmentation
> قبل از تقسیم train/test انجام شده و نسخه‌های augmented یک تصویر می‌توانند هم در train و هم در test بیفتند — به همین دلیل
> از آن دیتاست استفاده نکردیم؛ عدد دقت بالای آن گمراه‌کننده است، نه نشانه کیفیت بهتر.
>
> **جمع‌بندی:** برای این پروژه (نمونه اولیه / پایان‌نامه) این سطح از دقت کاملاً کافی و قابل قبول است، ولی این عدد را به‌عنوان
> «دقت تشخیص بالینی روی بیمار جدید» معرفی نکنید. برای اعتبارسنجی بالینی واقعی به دیتاستی با patient ID مجزا (مثل ADNI) و
> split در سطح بیمار نیاز است.


## ۸. Grad-CAM واقعی

این تابع دقیقاً همان منطقی است که در `apps/diagnosis/ml_inference.py` سمت جنگو استفاده می‌شود — اینجا فقط برای اطمینان از درست‌کار‌کردن آن، روی چند نمونه تست می‌کنیم.

In [ ]:
def make_gradcam_heatmap(img_array, model, base_layer_name="mobilenetv2_base", last_conv_name="Conv_1"):
    base = model.get_layer(base_layer_name)
    conv_layer = base.get_layer(last_conv_name)
    grad_model = tf.keras.models.Model(model.inputs, [conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        class_idx = tf.argmax(preds[0])
        class_channel = preds[:, class_idx]

    grads = tape.gradient(class_channel, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = tf.reduce_sum(conv_out * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(class_idx.numpy())


def show_gradcam_sample(image_path):
    img = tf.keras.utils.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = tf.keras.utils.img_to_array(img)
    batch = np.expand_dims(arr, axis=0)

    heatmap, class_idx = make_gradcam_heatmap(batch, model)
    heatmap_resized = np.array(
        tf.image.resize(heatmap[..., np.newaxis], (IMG_SIZE, IMG_SIZE))
    ).squeeze()

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(arr.astype("uint8"))
    axes[0].set_title(f"Original — pred: {CLASS_NAMES[class_idx]}")
    axes[0].axis("off")
    axes[1].imshow(arr.astype("uint8"))
    axes[1].imshow(heatmap_resized, cmap="jet", alpha=0.5)
    axes[1].set_title("Grad-CAM")
    axes[1].axis("off")
    plt.show()


# یک نمونه تصادفی از هر کلاس در دیتاست تست را نشان بده
import random
for cls in CLASS_NAMES:
    cls_dir = os.path.join(TEST_DIR, cls)
    sample_file = random.choice(os.listdir(cls_dir))
    show_gradcam_sample(os.path.join(cls_dir, sample_file))


## ۹. پیش‌نمایش خروجی به شکل سازگار با جنگو

قبل از ذخیره، مطمئن می‌شویم خروجی مدل دقیقاً همان ساختار دیکشنری‌ای است که تمپلیت `dashboard/result.html` انتظار دارد
(`result`, `result_label`, `confidence`, `probabilities`, `regions`, `risk_score`, `recommendation`).

In [ ]:
CLASS_LABELS_FA = {
    "NonDemented": "سالم (CN)",
    "VeryMildDemented": "اختلال بسیار خفیف (MCI)",
    "MildDemented": "اختلال خفیف (AD اولیه)",
    "ModerateDemented": "آلزایمر (AD)",
}
CLASS_COLORS = {
    "NonDemented": "#22c55e",
    "VeryMildDemented": "#f59e0b",
    "MildDemented": "#f97316",
    "ModerateDemented": "#ef4444",
}
RESULT_MAP = {
    "NonDemented": "CN",
    "VeryMildDemented": "MCI",
    "MildDemented": "AD",
    "ModerateDemented": "AD",
}

def preview_analysis_dict(image_path):
    img = tf.keras.utils.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = tf.keras.utils.img_to_array(img)
    batch = np.expand_dims(arr, axis=0)
    preds = model.predict(batch, verbose=0)[0]

    probabilities = [
        {
            "key": name,
            "label": CLASS_LABELS_FA[name],
            "value": round(float(preds[i]) * 100, 1),
            "color": CLASS_COLORS[name],
        }
        for i, name in enumerate(CLASS_NAMES)
    ]
    top_idx = int(np.argmax(preds))
    top_class = CLASS_NAMES[top_idx]
    return {
        "result": RESULT_MAP[top_class],
        "result_label": CLASS_LABELS_FA[top_class],
        "confidence": round(float(preds[top_idx]) * 100, 1),
        "probabilities": probabilities,
    }

sample_cls = CLASS_NAMES[0]
sample_dir = os.path.join(TEST_DIR, sample_cls)
sample_file = os.path.join(sample_dir, os.listdir(sample_dir)[0])
import pprint
pprint.pprint(preview_analysis_dict(sample_file))


## ۱۰. ذخیره مدل و دانلود برای پروژه جنگو

دو فایل ساخته می‌شود که باید داخل `apps/diagnosis/ml/` در پروژه جنگو کپی شوند.

In [ ]:
import json

model.save("alzheimer_model.keras")

with open("class_names.json", "w", encoding="utf-8") as f:
    json.dump(CLASS_NAMES, f, ensure_ascii=False, indent=2)

print("Saved: alzheimer_model.keras, class_names.json")
print("Class order:", CLASS_NAMES)


In [ ]:
from google.colab import files

files.download("alzheimer_model.keras")
files.download("class_names.json")


## راهنمای یکپارچه‌سازی با پروژه جنگو

1. دو فایل دانلود‌شده (`alzheimer_model.keras`, `class_names.json`) را داخل `apps/diagnosis/ml/` پروژه کپی کنید.
2. `tensorflow` را نصب کنید: `pip install -r requirements-ml.txt`
3. در `core/settings.py` مقدار `DEMO_MODE = False` را تنظیم کنید.
4. سرور جنگو را ری‌استارت کنید.

`apps/diagnosis/analyzer.py` و `apps/diagnosis/ml_inference.py` از قبل آماده‌اند تا همین دو فایل را به‌صورت خودکار بارگذاری کرده،
پیش‌بینی + Grad-CAM واقعی تولید کنند و دقیقاً همان ساختار دیکشنری DEMO_MODE را برگردانند — نیازی به تغییر تمپلیت یا view نیست.

**نکته دقت مدل:** اگر دقت تست کمتر از حد انتظار بود، اول تعداد epoch فاز فاین‌تیون را افزایش دهید یا `base_model.layers[:-30]` را به عدد بزرگ‌تر (مثلاً `-50`) تغییر دهید تا لایه‌های بیشتری فاین‌تیون شوند.
